# Courses 1–3 · What is Data Science, Tools, Methodology
These courses are mostly conceptual. This notebook practises the **tools** (Jupyter, Markdown, magics, Git) and walks the **10-stage methodology** on the practice data.
Every code line has a `#` comment explaining it.

## Part A · Jupyter essentials (Course 2)
| Key | Action |
|---|---|
| Shift+Enter | Run cell |
| Esc then A / B | New cell above / below |
| Esc then M / Y | Markdown / Code cell |
| Esc then D D | Delete cell |

This cell is **Markdown**: you can write *italics*, **bold**, lists, tables, and formulas like $y = mx + b$.
Double-click it to see the raw text.

In [1]:
# %timeit is a "line magic" (one % = applies to this line only).
# It runs the code many times and reports the average time: handy for comparing two approaches.
# sum(range(1000)) adds the numbers 0..999.
%timeit sum(range(1000))

19.7 µs ± 2.52 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [2]:
%%time
# %%time is a "cell magic" (two %% = applies to the whole cell; it must be the first line).
# It reports how long this whole cell took to run once.
total = 0                       # start a running total at zero
for i in range(1_000_000):      # loop over 0..999,999 (underscores just make big numbers readable)
    total += i                  # add i to the total (same as: total = total + i)
total                           # the last line's value is shown automatically as the cell output

CPU times: user 105 ms, sys: 1.96 ms, total: 107 ms
Wall time: 105 ms


499999500000

In [3]:
# Lines starting with ! are sent to the operating system's command line, not to Python.
!python --version
# ls lists the files in the current folder (use !dir on a Windows local install).
!ls

Python 3.9.12
00_setup.ipynb				09_machine_learning.ipynb
01_03_concepts_tools_methodology.ipynb	10_capstone_spacex.ipynb
04_python.ipynb				README.md
05_python_project_scraping_api.ipynb	applications.csv
06_sql.ipynb				centers.csv
07_data_analysis.ipynb			make_data.py
08_visualization.ipynb


## Part B · Git basics (Course 2)
Run these in a **Terminal** (File → New → Terminal), not in a notebook cell. Each line explained:
```bash
git config --global user.name "Your Name"         # tell Git who you are (once per computer)
git config --global user.email "you@example.com"  # your email appears on every commit
git init                                          # turn this folder into a Git repository
git add make_data.py 00_setup.ipynb               # stage files: choose what goes in the next snapshot
git commit -m "Add practice data"                 # save the snapshot with a message
git branch -M main                                # name the main branch "main"
git remote add origin https://github.com/<you>/ds-review-labs.git   # link to an empty GitHub repo
git push -u origin main                           # upload commits to GitHub
```

## Part C · The 10-stage methodology on real data (Course 3)
### Stage 1 · Business understanding
> Applicants wait too long for their ID. **Goal:** average processing under 6 days in every region, and fewer rejected applications.

### Stage 2 · Analytic approach
- *How long will it take?* → **regression** (predicts a number)
- *Will it be rejected?* → **classification** (predicts a category)

### Stages 3–4 · Data requirements and collection

In [ ]:
# os lets Python talk to the operating system (files, folders).
import os
# If the practice data file is not in this folder yet...
if not os.path.exists("applications.csv"):
    # ...run make_data.py to create applications.csv and centers.csv.
    %run make_data.py
# pandas = tables (nickname pd); numpy = fast math (nickname np). These nicknames are standard.
import pandas as pd, numpy as np

# Load the two tables into DataFrames.
apps = pd.read_csv("applications.csv")
centers = pd.read_csv("centers.csv")
# .columns lists the column names; .tolist() turns them into a plain Python list for printing.
print(apps.columns.tolist())

### Stage 5 · Data understanding: is the data good enough?

In [ ]:
# len() counts the rows.
print("Rows:", len(apps))
# .duplicated() marks each row that is an exact copy of an earlier row (True/False);
# .sum() counts the Trues (True counts as 1).
print("Duplicates:", apps.duplicated().sum())
# .isna() marks missing values; .sum() counts them per column.
missing = apps.isna().sum()
# Keep only the columns that actually have missing values (missing > 0 is a True/False filter).
print("Missing values:\n", missing[missing > 0])
# .unique() lists the distinct values in a column: spot the inconsistent "Walk-In " label.
print("Channel labels:", apps.channel.unique())

We found three quality problems (duplicates, missing ages, messy labels) → go to **Stage 6**.

In [ ]:
# Stage 6 · Data preparation, written as a "method chain": each step feeds the next.
df = (apps.drop_duplicates()                                   # 1. remove exact duplicate rows
          .assign(                                             # 2. assign() creates or replaces columns:
              channel=lambda d: d.channel.str.strip()          #    remove spaces at the start/end of the text,
                                         .str.lower(),         #    then make it lowercase ("Walk-In " -> "walk-in")
              age=lambda d: d.age.fillna(d.age.median()))      #    fill missing ages with the median age
          .merge(centers, on="center_id"))                     # 3. join center info (region, staff) by center_id
# "lambda d:" is a tiny unnamed function; d is the table at that point in the chain.
print(df.shape, df.channel.unique())                          # check: fewer rows, three clean labels

### Stages 7–8 · A first, simple model and its evaluation

In [ ]:
# Import the linear regression model and the function that splits data into train/test sets.
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# X = the inputs (features) the model learns from; y = the target it predicts.
X = df[["staff", "docs_complete", "biometric_quality"]]   # double brackets -> a table of 3 columns
y = df["processing_days"]                                  # single brackets -> one column (a Series)

# Keep 20% of rows aside as unseen "test" data; random_state=0 makes the split repeatable.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

# Create the model and .fit() it: it learns the best line through the training data.
m = LinearRegression().fit(X_tr, y_tr)

# .score() gives R² on the test data: 1.0 = perfect, 0 = no better than guessing the average.
print("R² on unseen data:", round(m.score(X_te, y_te), 3))
# .coef_ holds one learned weight per feature; zip() pairs each name with its weight.
print(dict(zip(X.columns, m.coef_.round(2))))

**Reading it:** each extra staff member cuts about 0.15 days; complete documents cut about 5 days. That answers Stage 1 with numbers a manager can act on.

### Stages 9–10 · Deployment and feedback
Deployment = a weekly report of slow regions. Feedback = compare predictions with real outcomes each month.

In [ ]:
# groupby("region") splits rows by region; ["processing_days"].mean() averages each group;
# round(1) keeps one decimal; sort_values(ascending=False) puts the slowest region first.
df.groupby("region")["processing_days"].mean().round(1).sort_values(ascending=False)

## ✍️ Your turn
1. Write a Stage-1 problem statement for an issue in your own work (one sentence with a number).
2. Which region misses the 6-day goal? What data would you collect next to explain why?

In [ ]:
# your answer here